In [22]:
import pandas as pd  # 데이터 처리
from pathlib import Path  # 경로 처리

# =========================
# 0) 파일 경로
# =========================
SRC_XLSX = Path("상품DB_251216.xlsx")  # 원본 엑셀 경로
OUT_7TABLES = Path("피프틴양식.xlsx")  # 7개 테이블 출력
OUT_MERGED = Path("상품통합.xlsx")  # 통합DB 출력

# =========================
# 1) 고정 마스터 (language)
# =========================
LANGUAGE_ROWS = [  # 언어 마스터 고정값
    ("01", "한국어", "ko"),  # 한국어
    ("02", "영어", "en"),  # 영어
    ("03", "태국어", "th"),  # 태국어
    ("04", "베트남어", "vi"),  # 베트남어
    ("05", "인도네시아어", "id"),  # 인도네시아어
]  # 언어 목록 끝

# =========================
# 2) 카테고리 매핑 (KO 기준)
#    - sub_name(하위명) -> (top_seq 2자리, sub_seq 3자리, top_name_ko)
# =========================
SUB_MAP = {  # 하위카테고리명 매핑
    "과일채소": ("01", "001", "과일채소"),  # 과일채소
    "고기완자": ("02", "001", "육류"),  # 고기완자
    "냉장햄": ("02", "002", "육류"),  # 냉장햄
    "부속고기": ("02", "003", "육류"),  # 부속고기
    "육류가공품": ("02", "004", "육류"),  # 육류가공품
    "원육": ("02", "005", "육류"),  # 원육
    "냉동닭(할랄)": ("02", "006", "육류"),  # 냉동닭
    "냉동식품": ("03", "001", "냉동식품"),  # 냉동식품
    "냉동수산": ("04", "001", "수산물"),  # 냉동수산
    "수산물가공품": ("04", "002", "수산물"),  # 수산가공
    "라이스페이퍼": ("05", "001", "주식류"),  # 라이스페이퍼
    "면류": ("05", "002", "주식류"),  # 면류
    "즉석밥": ("05", "003", "주식류"),  # 즉석밥
    "라면류": ("05", "004", "주식류"),  # 라면류
    "빵류": ("05", "005", "주식류"),  # 빵류
    "코코넛가공품": ("06", "001", "간식류"),  # 코코넛가공
    "디저트류": ("06", "002", "간식류"),  # 디저트
    "과자": ("06", "003", "간식류"),  # 과자
    "과일캔": ("06", "004", "간식류"),  # 과일캔
    "기타절임류": ("07", "001", "절임가공류"),  # 기타절임
    "피클": ("07", "002", "절임가공류"),  # 피클
    "죽순절임": ("07", "003", "절임가공류"),  # 죽순절임
    "젓갈": ("07", "004", "절임가공류"),  # 젓갈
    "피쉬소스": ("08", "001", "소스류"),  # 피쉬소스
    "칠리소스": ("08", "002", "소스류"),  # 칠리소스
    "소스류": ("08", "003", "소스류"),  # 기타소스
    "양념": ("08", "004", "소스류"),  # 양념
    "간장": ("08", "005", "소스류"),  # 간장
    "오일": ("08", "006", "소스류"),  # 오일
    "페이스트": ("09", "001", "조미료"),  # 페이스트
    "고추/고춧가루": ("09", "002", "조미료"),  # 고추류
    "조미료": ("09", "003", "조미료"),  # 조미료
    "커리": ("09", "004", "조미료"),  # 커리
    "향신료": ("09", "005", "조미료"),  # 향신료
    "화장품": ("10", "001", "미용제품"),  # 화장품
    "비누": ("10", "002", "미용제품"),  # 비누
    "샴푸": ("10", "003", "미용제품"),  # 샴푸
    "곡류": ("11", "001", "곡류"),  # 곡류
    "조리도구": ("12", "001", "주방제품"),  # 조리도구
    "음료": ("13", "001", "음료"),  # 음료
    "커피/티": ("13", "002", "음료"),  # 커피티
    "기타": ("14", "001", "기타"),  # 기타
}  # 매핑 끝

UNIT_CODE = {  # 옵션단위 코드
    "EA": "01",  # 낱개
    "PACK": "02",  # 팩
    "BUNDLE": "03",  # 번들
    "BOX": "04",  # 박스
}  # 옵션단위 끝

# =========================
# 3) 유틸 함수
# =========================
def zfill_num(s: pd.Series, n: int) -> pd.Series:  # 0채우기
    return s.astype("Int64").astype("string").str.zfill(n)  # 숫자->문자->zfill

def safe_col(df: pd.DataFrame, name: str) -> pd.Series:  # 컬럼 없으면 빈값
    return df[name] if name in df.columns else pd.Series([pd.NA] * len(df))  # 안전 반환

def rename_cols(df: pd.DataFrame, col_map: dict) -> pd.DataFrame:  # 컬럼명 바꾸기
    return df.rename(columns=col_map)  # 컬럼명 변경

# =========================
# 4) 원본 시트 읽기
# =========================
base = pd.read_excel(SRC_XLSX, sheet_name=0)  # 1시트(기본)
price = pd.read_excel(SRC_XLSX, sheet_name=1)  # 2시트(가격/다국어)

base["품목코드"] = base["품목코드"].astype("Int64")  # 품목코드 정리
price["품목코드"] = price["품목코드"].astype("Int64")  # 품목코드 정리

base["product_no"] = zfill_num(base["품목코드"], 10)  # 상품No(10자리)
price["product_no"] = zfill_num(price["품목코드"], 10)  # 상품No(10자리)

# =========================
# 5) “처음부터 상품명 붙여서” 조합용 기준 DF 만들기
#    - 원본 품목명(origin_product_name)을 여기서부터 유지
# =========================
base_cols = ["product_no"]  # 기본 컬럼
for col in ["품목명", "국가명", "과세/면세", "카테고리명", "단위"]:  # 필요한 컬럼들
    if col in base.columns:  # 있으면 추가
        base_cols.append(col)  # 컬럼 추가
df = price.merge(  # 가격시트에
    base[base_cols].copy(),  # 기본정보 붙이기
    on="product_no",  # 상품No로 조인
    how="left",  # 왼쪽 유지
)  # merge 끝

df["origin_product_name"] = safe_col(df, "품목명_x")  # 원본 상품명 고정 보관

# =========================
# 6) 카테고리 번호 생성 (상위 FK 4자리 통일)
# =========================
df["sub_name_ko"] = safe_col(df, "카테고리").fillna(df["카테고리명"])  # 하위카테고리명 후보

df["top_seq"] = df["sub_name_ko"].map(lambda x: SUB_MAP.get(x, (None, None, None))[0])  # 상위2자리
df["sub_seq"] = df["sub_name_ko"].map(lambda x: SUB_MAP.get(x, (None, None, None))[1])  # 하위3자리
df["top_name_ko"] = df["sub_name_ko"].map(lambda x: SUB_MAP.get(x, (None, None, None))[2])  # 상위명(ko)

df["top_category_no"] = ("01" + df["top_seq"].astype("string")).where(df["top_seq"].notna())  # 상위No(4자리)
df["sub_category_no"] = ("01" + df["top_seq"].astype("string") + df["sub_seq"].astype("string")).where(df["top_seq"].notna())  # 하위No(7자리)

# =========================
# 7) 7개 테이블 생성
# =========================

# 7-1) language
language = pd.DataFrame(LANGUAGE_ROWS, columns=["language_no", "language_name", "language_code"])  # 언어테이블

# 7-2) top_category (KO만 생성: 데이터에 등장한 것만)
top_category = (  # 상위카테고리
    df[["top_category_no", "top_name_ko"]].dropna().drop_duplicates()  # 중복 제거
    .assign(language_no="01")  # 한국어 고정
    .rename(columns={"top_name_ko": "category_name"})  # 컬럼명 맞춤
    [["top_category_no", "language_no", "category_name"]]  # 순서 고정
)  # top_category 끝

# 7-3) sub_category (KO만 생성: 데이터에 등장한 것만)
sub_category = (  # 하위카테고리
    df[["sub_category_no", "top_category_no", "sub_name_ko"]].dropna().drop_duplicates()  # 중복 제거
    .assign(language_no="01")  # 한국어 고정
    .rename(columns={"sub_name_ko": "sub_category_name"})  # 컬럼명 맞춤
    [["sub_category_no", "top_category_no", "language_no", "sub_category_name"]]  # 순서 고정
)  # sub_category 끝

# 7-4) product
product = pd.DataFrame()  # product 만들기
product["product_no"] = df["product_no"]  # 상품No
product["top_category_no"] = df["top_category_no"]  # 상위FK(4자리)
product["sub_category_no"] = df["sub_category_no"]  # 하위FK(7자리)
product["one_time_limit_yn"] = 0  # 기본 0
product["tax_free_yn"] = (df["과세/면세"] == "면세").astype("int")  # 면세면 1
product["wms_code"] = zfill_num(df["품목코드"], 8)  # 예시: 8자리
product["item_code"] = zfill_num(df["품목코드"], 8)  # 예시: 8자리
product["coupang_proxy_yn"] = 0  # 기본 0
product["safety_stock"] = pd.NA  # 원본 없으면 비움
product["temp_soldout_yn"] = 0  # 기본 0
product["timesale_start_dt"] = pd.NA  # 원본 없으면 비움
product["timesale_end_dt"] = pd.NA  # 원본 없으면 비움
product["timesale_discount_type"] = pd.NA  # 원본 없으면 비움
product["timesale_discount_value"] = pd.NA  # 원본 없으면 비움
product = product.drop_duplicates(subset=["product_no"])  # 상품 중복 제거

# 7-5) product_lang (여기서 “처음부터 상품명 붙이기” 핵심)
def make_product_lang(lang_no: str, name_col: str) -> pd.DataFrame:  # 언어별 상품정보 생성
    t = pd.DataFrame()  # 임시 DF
    t["product_no"] = df["product_no"]  # FK
    t["language_no"] = lang_no  # 언어 FK
    if name_col == "품목명":  # 원본 품목명 쓰는 경우
        t["product_name"] = df["origin_product_name"]  # 원본 상품명 사용
    else:  # 특정 컬럼 쓰는 경우
        t["product_name"] = safe_col(df, name_col)  # 해당 언어컬럼 사용
    t["origin"] = safe_col(df, "국가명")  # 원산지
    t["description"] = pd.NA  # 없으면 비움
    t["notice"] = pd.NA  # 없으면 비움
    t = t[t["product_name"].notna()]  # 이름 없는 행 제거
    t["product_lang_no"] = t["product_no"] + t["language_no"]  # PK(단순합성)
    return t[["product_lang_no", "product_no", "language_no", "product_name", "origin", "description", "notice"]]  # 반환

pl_ko = make_product_lang("01", "품목명")  # KO는 원본 품목명으로 고정
pl_en = make_product_lang("02", "상품 영어명")  # EN
pl_th = make_product_lang("03", "태국")  # TH
pl_vi = make_product_lang("04", "베트남")  # VI
pl_id = make_product_lang("05", "인도네시아")  # ID

product_lang = pd.concat([pl_ko, pl_en, pl_th, pl_vi, pl_id], ignore_index=True)  # 합치기

# 7-6) product_option
opt_rows = []  # 옵션 행 리스트
for _, r in df.iterrows():  # 한줄씩
    pno = r["product_no"]  # 상품No
    base_unit = str(r["단위"]) if pd.notna(r.get("단위")) else "EA"  # 기본단위
    base_code = UNIT_CODE.get(base_unit, "01")  # 코드 변환
    opt_rows.append((pno + base_code, pno, base_code, r.get("규격정보"), 1, r.get("순번"), pd.NA))  # 기본옵션
    if pd.notna(r.get("팩품목코드")):  # 팩 있으면
        opt_rows.append((pno + "02", pno, "02", r.get("규격정보"), r.get("팩품목환산수량\n(1팩에 몇 개)"), pd.NA, pd.NA))  # 팩옵션
    if pd.notna(r.get("번들품목코드")):  # 번들 있으면
        opt_rows.append((pno + "03", pno, "03", r.get("규격정보"), r.get("번들품목환산수량\n(번들에 몇 개)"), pd.NA, pd.NA))  # 번들옵션
    if pd.notna(r.get("박스품목 환산수량")):  # 박스 있으면
        opt_rows.append((pno + "04", pno, "04", r.get("규격정보"), r.get("박스품목 환산수량"), pd.NA, pd.NA))  # 박스옵션

product_option = pd.DataFrame(  # 옵션 DF
    opt_rows,  # 데이터
    columns=["product_option_no", "product_no", "option_unit", "weight", "sale_unit", "max_buy_qty", "temp_soldout_text"],  # 컬럼
)  # DF 생성
product_option = product_option.drop_duplicates(subset=["product_option_no"])  # 중복 제거

# 7-7) product_option_lang_price (언어별 가격 “전부”)
def add_price_row(rows: list, opt_no: str, lang_no: str, inbound, cost, sale):  # 가격행 추가
    rows.append((opt_no + lang_no, lang_no, opt_no, inbound, cost, sale))  # PK(옵션+언어) 생성

price_rows = []  # 가격 행 리스트
for _, r in df.iterrows():  # 한줄씩
    pno = r["product_no"]  # 상품No
    base_unit = str(r["단위"]) if pd.notna(r.get("단위")) else "EA"  # 기본단위
    base_code = UNIT_CODE.get(base_unit, "01")  # 코드
    ea_no = pno + base_code  # 기본 옵션No
    for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 전부
        add_price_row(price_rows, ea_no, lang_no, r.get("입고단가"), r.get("낱개단가"), r.get("낱개단가"))  # 낱개 가격
    if pd.notna(r.get("팩품목코드")):  # 팩이면
        pack_no = pno + "02"  # 팩 옵션No
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 전부
            add_price_row(price_rows, pack_no, lang_no, r.get("입고단가"), r.get("팩단가"), r.get("팩단가"))  # 팩 가격
    if pd.notna(r.get("번들품목코드")):  # 번들이면
        bun_no = pno + "03"  # 번들 옵션No
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 전부
            add_price_row(price_rows, bun_no, lang_no, r.get("입고단가"), r.get("번들단가"), r.get("번들단가"))  # 번들 가격
    if pd.notna(r.get("박스품목 환산수량")):  # 박스면
        box_no = pno + "04"  # 박스 옵션No
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 전부
            add_price_row(price_rows, box_no, lang_no, r.get("입고단가"), r.get("박스단가"), r.get("박스단가"))  # 박스 가격

product_option_lang_price = pd.DataFrame(  # 가격 DF
    price_rows,  # 데이터
    columns=["option_lang_no", "language_no", "product_option_no", "inbound_price", "cost_price", "sale_price"],  # 컬럼
)  # DF 생성
product_option_lang_price = product_option_lang_price.drop_duplicates(subset=["product_option_no", "language_no"])  # 중복 제거

# =========================
# 8) 통합DB 생성 (옵션+언어 1행, 언어별 가격 전부)
#    - 여기에도 원본 상품명(origin_product_name) 포함
# =========================
merged = product_option_lang_price.merge(product_option, on="product_option_no", how="left")  # 가격+옵션
merged = merged.merge(product, on="product_no", how="left")  # +상품
merged = merged.merge(product_lang, on=["product_no", "language_no"], how="left")  # +언어별 상품
merged = merged.merge(df[["product_no", "origin_product_name"]].drop_duplicates(), on="product_no", how="left")  # +원본 상품명

# =========================
# 8-1) 컬럼명 한글로 변경(여기서 적용)
# =========================
language = rename_cols(language, {  # language 컬럼명 변경
    "language_no": "언어 No",  # 언어 PK
    "language_name": "언어명",  # 언어명
    "language_code": "언어 코드",  # 언어코드
})  # language 끝

top_category = rename_cols(top_category, {  # top_category 컬럼명 변경
    "top_category_no": "카테고리 No (PK)",  # 카테고리 PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "category_name": "카테고리명",  # 카테고리명
})  # top_category 끝

sub_category = rename_cols(sub_category, {  # sub_category 컬럼명 변경
    "sub_category_no": "하위 카테고리 No (PK)",  # 하위 PK
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "language_no": "언어 No (FK)",  # 언어 FK
    "sub_category_name": "하위 카테고리명",  # 하위명
})  # sub_category 끝

product = rename_cols(product, {  # product 컬럼명 변경
    "product_no": "상품 No (PK)",  # 상품 PK
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "sub_category_no": "하위 카테고리 No (FK)",  # 하위 FK
    "one_time_limit_yn": "1회 구매 제한 상품 여부 (0-X, 1-O)",  # 제한 여부
    "tax_free_yn": "면세 상품 여부 (0-X, 1-O)",  # 면세 여부
    "wms_code": "WMS 코드",  # WMS 코드
    "item_code": "품목 코드",  # 품목 코드
    "coupang_proxy_yn": "쿠팡 대리구매 상품 여부 (0-X, 1-O)",  # 쿠팡 여부
    "safety_stock": "안전재고",  # 안전재고
    "temp_soldout_yn": "임시품절 여부 (0-X, 1-O)",  # 임시품절
    "timesale_start_dt": "타임세일 시작일시",  # 타임세일 시작
    "timesale_end_dt": "타임세일 종료일시",  # 타임세일 종료
    "timesale_discount_type": "타임세일 할인 유형 (percent => 정률, price => 정액)",  # 할인유형
    "timesale_discount_value": "타임세일 할인값",  # 할인값
})  # product 끝

product_lang = rename_cols(product_lang, {  # product_lang 컬럼명 변경
    "product_lang_no": "언어별 상품정보 No (PK)",  # 언어별 PK
    "product_no": "상품 No (FK)",  # 상품 FK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_name": "상품명",  # 상품명
    "origin": "원산지",  # 원산지
    "description": "상세 설명",  # 상세설명
    "notice": "상품 정보고시",  # 고시
})  # product_lang 끝

product_option = rename_cols(product_option, {  # product_option 컬럼명 변경
    "product_option_no": "상품 옵션 No (PK)",  # 옵션 PK
    "product_no": "상품 No (FK)",  # 상품 FK
    "option_unit": "옵션 단위(EA, BUNDLE, PACK, BOX)",  # 옵션단위
    "weight": "무게",  # 무게
    "sale_unit": "판매 단위",  # 판매단위
    "max_buy_qty": "최대 구매 수량",  # 최대수량
    "temp_soldout_text": "임시 품절여부",  # 임시품절 텍스트
})  # product_option 끝

product_option_lang_price = rename_cols(product_option_lang_price, {  # 가격 컬럼명 변경
    "option_lang_no": "언어별 상품 옵션정보 No (PK)",  # PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_option_no": "상품 옵션 No (FK)",  # 옵션 FK
    "inbound_price": "입고가",  # 입고가
    "cost_price": "원가",  # 원가
    "sale_price": "판매가",  # 판매가
})  # 가격 끝

merged = rename_cols(merged, {  # merged 컬럼명 변경
    "origin_product_name": "원본 상품명",  # 원본명
    "option_lang_no": "언어별 상품 옵션정보 No (PK)",  # PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_option_no": "상품 옵션 No (FK)",  # 옵션 FK
    "product_no": "상품 No (FK)",  # 상품 FK
    "inbound_price": "입고가",  # 입고가
    "cost_price": "원가",  # 원가
    "sale_price": "판매가",  # 판매가
    "option_unit": "옵션 단위(EA, BUNDLE, PACK, BOX)",  # 옵션단위
    "weight": "무게",  # 무게
    "sale_unit": "판매 단위",  # 판매단위
    "max_buy_qty": "최대 구매 수량",  # 최대수량
    "temp_soldout_text": "임시 품절여부",  # 임시품절 텍스트
    "product_name": "상품명",  # 상품명
    "origin": "원산지",  # 원산지
    "description": "상세 설명",  # 상세설명
    "notice": "상품 정보고시",  # 고시
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "sub_category_no": "하위 카테고리 No (FK)",  # 하위 FK
    "one_time_limit_yn": "1회 구매 제한 상품 여부 (0-X, 1-O)",  # 제한 여부
    "tax_free_yn": "면세 상품 여부 (0-X, 1-O)",  # 면세 여부
    "wms_code": "WMS 코드",  # WMS 코드
    "item_code": "품목 코드",  # 품목 코드
    "coupang_proxy_yn": "쿠팡 대리구매 상품 여부 (0-X, 1-O)",  # 쿠팡 여부
    "safety_stock": "안전재고",  # 안전재고
    "temp_soldout_yn": "임시품절 여부 (0-X, 1-O)",  # 임시품절
    "timesale_start_dt": "타임세일 시작일시",  # 타임세일 시작
    "timesale_end_dt": "타임세일 종료일시",  # 타임세일 종료
    "timesale_discount_type": "타임세일 할인 유형 (percent => 정률, price => 정액)",  # 할인유형
    "timesale_discount_value": "타임세일 할인값",  # 할인값
})  # merged 끝

# =========================
# 9) 엑셀 저장 (7테이블 / 통합DB)
# =========================
with pd.ExcelWriter(OUT_7TABLES, engine="openpyxl") as w:  # 7테이블 저장
    language.to_excel(w, sheet_name="language", index=False)  # 1) language
    top_category.to_excel(w, sheet_name="top_category", index=False)  # 2) top_category
    sub_category.to_excel(w, sheet_name="sub_category", index=False)  # 3) sub_category
    product.to_excel(w, sheet_name="product", index=False)  # 4) product
    product_lang.to_excel(w, sheet_name="product_lang", index=False)  # 5) product_lang
    product_option.to_excel(w, sheet_name="product_option", index=False)  # 6) product_option
    product_option_lang_price.to_excel(w, sheet_name="product_option_lang_price", index=False)  # 7) product_option_lang_price

with pd.ExcelWriter(OUT_MERGED, engine="openpyxl") as w:  # 통합DB 저장
    merged.to_excel(w, sheet_name="merged_all", index=False)  # 통합 시트 저장

print("완료:", OUT_7TABLES, OUT_MERGED)  # 결과 출력


완료: 피프틴양식.xlsx 상품통합.xlsx


In [16]:
print(price.columns.tolist())  # 2시트 컬럼명 전체 확인

['순번', '품목코드', '품목명', '단위', '달랏\n품목코드', '달랏\n상품명', '수입시 영어명', '상품 영어명', '원산지', '브랜드', '카테고리', '태국', '베트남', '인도네시아', '입고단가', '낱개단가', '팩품목코드', '연결품목단위2', '팩품목환산수량\n(1팩에 몇 개)', '팩단가', '번들품목코드', '번들품목단위', '번들품목환산수량\n(번들에 몇 개)', '번들단가', '대표품목코드', '박스품목단위', '박스품목 환산수량', '박스단가', 'product_no']


In [20]:
print([c for c in df.columns if "품목명" in c])  # 품목명 관련 컬럼 확인  # 디버그

['품목명_x', '품목명_y']
